# Avoiding model refits in leave-one-out cross-validation with moment matching

## Using PyMC and arviz-stats

This notebook demonstrates how to improve the Monte Carlo sampling accuracy of leave-one-out cross-validation using moment matching with PyMC models and the arviz-stats package. The methodology helps when Pareto $k$ diagnostics indicate problems with importance sampling accuracy.

The methodology presented is based on:

* Paananen, T., Piironen, J., Buerkner, P.-C., Vehtari, A. (2021). Implicitly Adaptive Importance Sampling. Statistics and Computing. 31(2). [arXiv:1906.08850](https://arxiv.org/abs/1906.08850)

* Vehtari, A., Gelman, A., and Gabry, J. (2017). Practical Bayesian model evaluation using leave-one-out cross-validation and WAIC. Statistics and Computing. 27(5), 1413--1432. [arXiv:1507.04544](https://arxiv.org/abs/1507.04544)

* Vehtari, A., Simpson, D., Gelman, A., Yao, Y., and Gabry, J. (2024). Pareto smoothed importance sampling. Journal of Machine Learning Research, 25(72):1-58.

## Setup and Imports

In [ ]:
import arviz as az
import numpy as np
import pandas as pd
import pymc as pm
import xarray as xr
from scipy import stats

from arviz_stats.loo import loo, loo_moment_match

az.style.use("arviz-doc")

## Example: Eradication of Roaches

We'll use the same roaches dataset as in the original vignette. This is a Poisson regression problem modeling the number of roaches caught in apartment buildings.

In [ ]:
roaches = pd.read_csv('roaches.csv')
roaches['roach1'] = np.sqrt(roaches['roach1'])

y = roaches['y'].values
X = roaches[['roach1', 'treatment', 'senior']].values
offset = np.log(roaches['exposure2'].values)

n, k = X.shape

print(f"Number of observations: {n}")
print(f"Number of predictors: {k}")
print(roaches.head())

## Building the Model

We'll create a Poisson regression model equivalent to the Stan model in the original vignette:

$$y_i \sim \text{Poisson}(\exp(X_i \beta + \alpha + \text{offset}_i))$$

With priors:
- $\beta_j \sim \text{Normal}(0, 2.5)$ for $j = 1, ..., k$
- $\alpha \sim \text{Normal}(0, 5)$

In [3]:
with pm.Model() as model:
    X_data = pm.Data('X', X)
    y_data = pm.Data('y', y)
    offset_data = pm.Data('offset', offset)

    intercept = pm.Normal('intercept', mu=0, sigma=5)
    beta = pm.Normal('beta', mu=0, sigma=2.5, shape=k)

    eta = pm.math.dot(X_data, beta) + intercept + offset_data

    y_obs = pm.Poisson('y_obs', mu=pm.math.exp(eta), observed=y_data)

## Fitting the Model

In [ ]:
with model:
    idata = pm.sample(
        draws=1000,
        chains=4,
        random_seed=315,
        idata_kwargs={'log_likelihood': True, 'include_transformed': True}
    )

In [ ]:
idata

## Initial PSIS-LOO-CV Analysis

Let's compute the initial LOO estimates and check for problematic observations:

In [ ]:
loo_orig = loo(idata, pointwise=True)
loo_orig

## Moment Matching for PyMC Models

To use moment matching with PyMC models, we need to:

1. Extract the unconstrained (transformed) parameters from the posterior
2. Use PyMC's compiled log probability function
3. Define a function to compute the pointwise log likelihood using PyMC's distributions

Since our model has no transformed parameters (all parameters are unconstrained), we can use the regular posterior values directly.

In [ ]:
posterior = idata.posterior

upars = xr.DataArray(
    np.concatenate([
        posterior['intercept'].values[..., None],
        posterior['beta'].values
    ], axis=-1),
    dims=['chain', 'draw', 'param'],
    coords={'chain': posterior.chain, 'draw': posterior.draw}
)

print(f"Unconstrained parameters shape: {upars.shape}")

## Define Helper Functions for Moment Matching

We need two key functions:
1. `log_prob_upars_fn`: Computes log posterior probability
2. `log_lik_i_upars_fn`: Computes log likelihood for observation i

In [8]:
compiled_logp = model.compile_logp()

def log_prob_upars(upars):
    """Compute log posterior probability for all samples in upars."""
    n_chains, n_draws = upars.shape[:2]
    logp_values = np.empty((n_chains, n_draws))

    point = model.initial_point()

    for chain_idx in range(n_chains):
        for draw_idx in range(n_draws):
            upars_values = upars.isel(chain=chain_idx, draw=draw_idx).values

            point['intercept'] = upars_values[0]
            point['beta'] = upars_values[1:]

            logp_values[chain_idx, draw_idx] = compiled_logp(point)

    return xr.DataArray(
        logp_values,
        dims=["chain", "draw"],
        coords={"chain": upars.coords["chain"], "draw": upars.coords["draw"]}
    )

def log_lik_i_upars(upars, i):
    """Compute log likelihood for observation i."""
    upars_np = upars.values

    intercept_values = upars_np[:, :, 0]
    beta_values = upars_np[:, :, 1:]

    eta_i = intercept_values + np.einsum('ijk,k->ij', beta_values, X[i]) + offset[i]
    mu_i = np.exp(eta_i)

    loglik_values = stats.poisson.logpmf(y[i], mu_i)

    return xr.DataArray(
        loglik_values,
        dims=["chain", "draw"],
        coords={"chain": upars.coords["chain"], "draw": upars.coords["draw"]}
    )

In [ ]:
print("Testing log_prob_upars:")
test_log_prob = log_prob_upars(upars)
print(f"Shape: {test_log_prob.shape}")
print(f"Mean log prob: {test_log_prob.mean().values:.2f}")

print("\nTesting log_lik_i_upars for observation 0:")
test_log_lik = log_lik_i_upars(upars, 0)
print(f"Shape: {test_log_lik.shape}")
print(f"Mean log lik: {test_log_lik.mean().values:.2f}")

## Apply Moment Matching

Now we can use the `loo_moment_match` function from arviz-stats to improve the LOO estimates for problematic observations:

In [ ]:
loo_orig

In [ ]:
loo_mm = loo_moment_match(
    idata,
    loo_orig,
    upars=upars,
    log_prob_upars_fn=log_prob_upars,
    log_lik_i_upars_fn=log_lik_i_upars,
    var_name='y_obs',
    split=False,
    cov=True,
    pointwise=True
)

In [ ]:
loo_mm

In [ ]:
loo_mm.p_loo_i